# Conditional Branching

Inspect one graph at a time to understand:
- which nodes are local to the graph
- which values are inherited from upstream state
- where `branch` logic and `active_when` logic are used
- which treatment endpoints the graph can emit


In [1]:
from pathlib import Path
import sys
import random

import pandas as pd
from IPython.display import HTML, Markdown, display

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

from scoop.configuration.config_loader import load_profile, load_profile_manifest, load_routed_profile
from scoop.utils.document_generation import make_mermaid, mermaid_html


2026-04-05 15:04:59,877 | DEBUG   | scoop | SCOOP logging initialized


In [2]:
profile_id = "nsclc"
graph_name = "nsclc_stg3_unresectable"

manifest = load_profile_manifest(profile_id, base_dir=CONFIG_ROOT)
runtime = load_routed_profile(profile_id, base_dir=CONFIG_ROOT)
ctx = load_profile(profile_id, graph_name, base_dir=CONFIG_ROOT)

display(Markdown(f"## Profile `{profile_id}` / Graph `{graph_name}`"))
display(pd.DataFrame([{
    "profile_id": manifest.profile_id,
    "default_graph": manifest.default_graph,
    "entry_graph": manifest.entry_graph,
    "selected_graph": graph_name,
    "available_graphs": ", ".join(sorted(manifest.graphs)),
}] ))

display(Markdown("## Pathway Mapping"))
display(pd.DataFrame([
    {"pathway": pathway, "tail_graph": tail_graph}
    for pathway, tail_graph in runtime.pathway_to_graph.items()
]).sort_values(["pathway"]).reset_index(drop=True))


## Profile `nsclc` / Graph `nsclc_stg3_unresectable`

,profile_id,default_graph,entry_graph,selected_graph,available_graphs
0,nsclc,nsclc,nsclc,nsclc_stg3_unresectable,"nsclc, nsclc_stg1, nsclc_stg23_resectable, nsc..."


## Pathway Mapping

,pathway,tail_graph
0,nsclc_stage1,nsclc_stg1
1,nsclc_stage23_resectable,nsclc_stg23_resectable
2,nsclc_stage3_unresectable,nsclc_stg3_unresectable
3,nsclc_stage4_driver,nsclc_stg4_driver
4,nsclc_stage4_nodrivers,nsclc_stg4_nodrivers


In [3]:
def node_summary_df(graph):
    rows = []
    for name, node in graph.nodes.items():
        rows.append({
            "node": name,
            "type": node.type,
            "active_when": bool(getattr(node, "active_when", None)),
            "labels": ", ".join(getattr(node, "labels", []) or []),
        })
    return pd.DataFrame(rows).sort_values(["type", "node"]).reset_index(drop=True)

def branch_rules_df(graph):
    rows = []
    for name, node in graph.nodes.items():
        if node.type != "branch":
            continue
        for idx, rule in enumerate(getattr(node, "rules", []) or [], start=1):
            cond = rule.get("if", {})
            rows.append({
                "branch_node": name,
                "rule_index": idx,
                "if": cond,
                "then": rule.get("then"),
                "default": getattr(node, "default", None),
            })
    return pd.DataFrame(rows)

def active_when_df(graph):
    rows = []
    for name, node in graph.nodes.items():
        active_when = getattr(node, "active_when", None) or {}
        for dep, value in active_when.items():
            rows.append({
                "node": name,
                "depends_on": dep,
                "required_value": value,
            })
    return pd.DataFrame(rows)

def inherited_dependencies_df(graph):
    local_nodes = set(graph.nodes)
    refs = []
    for name, node in graph.nodes.items():
        for dep in (getattr(node, "active_when", None) or {}):
            if dep not in local_nodes:
                refs.append({"node": name, "inherited_from": dep, "reason": "active_when"})
        if node.type == "branch":
            for rule in getattr(node, "rules", []) or []:
                for dep in rule.get("if", {}):
                    if dep not in local_nodes:
                        refs.append({"node": name, "inherited_from": dep, "reason": "branch_rule"})
    if not refs:
        return pd.DataFrame(columns=["node", "inherited_from", "reason"])
    return pd.DataFrame(refs).drop_duplicates().sort_values(["reason", "node", "inherited_from"]).reset_index(drop=True)

display(Markdown("## Node Summary"))
display(node_summary_df(ctx.graph))

display(Markdown("## Branch Rules"))
display(branch_rules_df(ctx.graph))

display(Markdown("## `active_when` Dependencies"))
display(active_when_df(ctx.graph))

display(Markdown("## Inherited Upstream Dependencies"))
display(inherited_dependencies_df(ctx.graph))


## Node Summary

,node,type,active_when,labels
0,condition_episode.crt_eligible,branch,False,
1,condition_episode.sequential_crt_eligible,branch,False,
2,condition_occurrence.egfr,branch,False,
3,treatment_episode.concurrent_crt,categorical,True,conc_crt
4,treatment_episode.concurrent_crt_adjuvant_durv...,categorical,True,conc_crt_adj_durvalumab
5,treatment_episode.concurrent_crt_adjuvant_osim...,categorical,True,conc_crt_adj_osimertinib
6,treatment_episode.radiotherapy,categorical,True,rt
7,treatment_episode.sequential_crt,categorical,True,seq_crt
8,treatment_episode.sequential_crt_adjuvant_durv...,categorical,True,seq_crt_adj_durvalumab
9,treatment_episode.sequential_crt_adjuvant_osim...,categorical,True,seq_crt_adj_osimertinib


## Branch Rules

,branch_node,rule_index,if,then,default
0,condition_occurrence.egfr,1,{'condition_occurrence.biomarker': 'egfr'},not_wt,wt
1,condition_episode.crt_eligible,1,{'person.performance_status_v_frail': 'not_v_f...,eligible,eligible
2,condition_episode.crt_eligible,2,{'person.performance_status_v_frail': 'v_frail'},ineligible,eligible
3,condition_episode.sequential_crt_eligible,1,{'person.performance_status': 'ps2'},eligible,ineligible


## `active_when` Dependencies

,node,depends_on,required_value
0,treatment_episode.radiotherapy,condition_episode.crt_eligible,ineligible
1,treatment_episode.sequential_crt,condition_episode.sequential_crt_eligible,eligible
2,treatment_episode.sequential_crt,condition_occurrence.egfr,wt
3,treatment_episode.sequential_crt,condition_occurrence.pdl1_binary,absent
4,treatment_episode.concurrent_crt,condition_episode.crt_eligible,eligible
5,treatment_episode.concurrent_crt,condition_episode.sequential_crt_eligible,ineligible
6,treatment_episode.concurrent_crt,condition_occurrence.egfr,wt
7,treatment_episode.concurrent_crt,condition_occurrence.pdl1_binary,absent
8,treatment_episode.sequential_crt_adjuvant_osim...,condition_episode.sequential_crt_eligible,eligible
9,treatment_episode.sequential_crt_adjuvant_osim...,condition_occurrence.egfr,not_wt


## Inherited Upstream Dependencies

,node,inherited_from,reason
0,treatment_episode.concurrent_crt,condition_occurrence.pdl1_binary,active_when
1,treatment_episode.concurrent_crt_adjuvant_durv...,condition_occurrence.pdl1_binary,active_when
2,treatment_episode.sequential_crt,condition_occurrence.pdl1_binary,active_when
3,treatment_episode.sequential_crt_adjuvant_durv...,condition_occurrence.pdl1_binary,active_when
4,condition_episode.crt_eligible,person.performance_status_v_frail,branch_rule
5,condition_episode.sequential_crt_eligible,person.performance_status,branch_rule
6,condition_occurrence.egfr,condition_occurrence.biomarker,branch_rule


In [6]:
display(Markdown("## Mermaid View"))
HTML(ctx.graph.to_mermaid())


## Mermaid View

In [ ]:
pathways_for_graph = [p for p, g in runtime.pathway_to_graph.items() if g == graph_name]
if not pathways_for_graph:
    raise RuntimeError(f"No manifest pathway routes to graph {graph_name!r}")
target_pathway = pathways_for_graph[0]

random.seed(17)
for _ in range(5000):
    routed = runtime.sample_routed_patient()
    if routed.pathway == target_pathway:
        break
else:
    raise RuntimeError(f"Did not sample pathway {target_pathway!r} within 5000 draws")

display(Markdown(f"## Sample Routed Patient For `{target_pathway}`"))
display(pd.DataFrame([{
    "pathway": routed.pathway,
    "tail_graph_name": routed.tail_graph_name,
    "outcome_error": routed.outcome_error,
    "primary_category_id": routed.graph_outcome.primary_category_id if routed.graph_outcome else None,
    "active_endpoints": routed.graph_outcome.active_endpoints if routed.graph_outcome else tuple(),
}]))

interesting_keys = [
    key for key in routed.final_sample
    if key.startswith("person.") or key.startswith("condition_occurrence.") or key.startswith("condition_episode.") or key.startswith("treatment_episode.")
]
interesting_rows = [
    {"field": key, "value": routed.final_sample[key]}
    for key in interesting_keys
    if routed.final_sample[key] is not None
]
display(pd.DataFrame(interesting_rows))
